# Tools 

In [1]:
from openai import OpenAI
import gradio as gr
from ollama import Client
import json



c:\Users\LOG IN\Desktop\llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ollama_base_url = 'http://localhost:11434/v1'
# model = 'llama3.2:3b'
model = 'qwen2.5:3b'
openai = OpenAI()
ollama = OpenAI(base_url=ollama_base_url,api_key='ollama')
client = Client(host=ollama_base_url)

In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [13]:
# create function which do task and call using gradio 
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content



In [14]:

gr.ChatInterface(
    fn=chat,
   
).launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


# Now lets use the tools 
 1.first we create an data------------------
 2.when user prompt we make first llm call then -----------------
 3.then we use the tools which check this data -----------------
 4.we call llm again with this imformation from tools then it gives result

In [4]:
# create an hardcoded data 
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

# create an function which gives a info from checking this data

def get_ticket_price(destination_city):
    print(f"tool called for {destination_city} city")
    price = ticket_prices.get(destination_city.lower(),'Unknown ticket price')
    return f" the price of a ticket to {destination_city} is {price}"


In [21]:
res = get_ticket_price("Berlin")
print(res)

tool called for Berlin city
 the price of a ticket to Berlin is $499


In [5]:
# There's a particular dictionary structure that's required to describe our function:
price_function = {
    'name':'get_ticket_price',
    "description": "Get the price of a return ticket to the destination city.",
    'parameters':{
        'type':'object',
        'properties':{
            "destination_city" : {
                'type':'string',
                "description": "The city that the customer wants to travel to",
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }

}

In [6]:
tools = [{'type':'function','function':price_function}]

Now lets call the model 

In [ ]:
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # call one 
    response = ollama.chat.completions.create(model=model,messages= messages,tools=tools)
    print(response.choices[0])
    if response.choices[0].finish_reason == 'tool_calls':
        user_message_response_details = response.choices[0].message # inside thus we get tool_call details 
        res_from_tool_function= handel_tool_call_process(user_message_response_details)
        messages.append(user_message_response_details)
        messages.append(res_from_tool_function) # this response becomes prompt to model it reads and based on taht gives output
    for message in messages:
        print(f'messages :- {message}')
# call 2nd 
    response = ollama.chat.completions.create(model = model ,messages=messages)
        # print(f'res_from_tool_function {res_from_tool_function}')
    return response.choices[0].message.content    

in first call we simple pass prompt and system message to model the  based on prompt messages it see should i need any tool or not 
then if yes the we get first call response in which we get tool_call details and the we pass our function which does the tool call process and receive details 
then we send that details in message to model then we call model again 

2nd call in this model receive details which has role = tool and content 
teen gives response,
user : what is the price for londen 
model : The price for a ticket to London is $799.
user: now what about paris price  
model :The price for a ticket to Paris is $899.

In [24]:
# we will wright functo to do this tool process manually later al this done by libray
# tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_1nh2cin0', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', index=0)]

def handel_tool_call_process(user_message):
    # print(f'user_message from tool call process {user_message}')
    tool_call = user_message.tool_calls[0]
    if tool_call.function.name == 'get_ticket_price':
        get_argument=json.loads(tool_call.function.arguments) # get argument and we pass to function to get result
        # print(f'argument{get_argument}') argument{'destination_city': 'london'}
        city = get_argument.get('destination_city') # londen
        price_details = get_ticket_price(city)
        # create the structure like messages because we are send this into massages which model receives 
        response = {
            'role':'tool',
            'content':price_details,
             'tool_call_id':tool_call.id
        }
    return response


In [ ]:
gr.ChatInterface(
    chat
).launch()

* Running on local URL:  http://127.0.0.1:7901
* To create a public link, set `share=True` in `launch()`.


Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? Are you looking for information about booking tickets or finding prices for your travel plans?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))
messages :- {'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
messages :- {'role': 'user', 'content': 'helo'}
Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_y2comkpl', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', index=0)]))
tool called for London city
messag

# Let's make couple of improvements 


Handel multiple tool cals in 1 response , 
Handel multiple tool calls 1 after another 

when i am asking for price of 2 destination_city it give only for first one because we have handle one 1 tool call above by if condition now handle multiple 

In [9]:
# kets update first this function above we are checking only for one tool_call now we wright for loop to chekc for multiple 
def handel_tool_call_process_improved(user_message):
    # print(f'user_message from tool call process {user_message}')
    response = []
    for tool_call in user_message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            get_argument=json.loads(tool_call.function.arguments) # get argument and we pass to function to get result
            # print(f'argument{get_argument}') argument{'destination_city': 'london'}
            city = get_argument.get('destination_city') # londen
            price_details = get_ticket_price(city)
            # create the structure like messages because we are send this into massages which model receives 
            response.append({
                'role':'tool',
                'content':price_details,
                'tool_call_id':tool_call.id
            })
    return response

In [ ]:
# lets update our chat function insted of if we use while loop 


def chat_improved(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = ([{"role": "system", "content": system_message}] + history+ [{"role": "user", "content": message}])

    # First model call
    response = ollama.chat.completions.create(model=model,messages=messages,tools=tools)

    # Keep looping until no more tool calls
    while response.choices[0].finish_reason == "tool_calls":
        print("Model requested tool call")
        assistant_message = response.choices[0].message
        tool_responses = handel_tool_call_process_improved(assistant_message)

        # Add assistant message
        messages.append(assistant_message)
        # Add ALL tool responses
        messages.extend(tool_responses)

        print("\nMessages being sent back:\n")
        for i, msg in enumerate(messages):
            print(i, msg)

        # Ask model again
        response = ollama.chat.completions.create(model=model,messages=messages,tools=tools)

    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat_improved).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Model requested tool call
tool called for London city

Messages being sent back:

0 {'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
1 {'role': 'user', 'content': [{'text': 'hello', 'type': 'text'}]}
2 {'role': 'assistant', 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}]}
3 {'role': 'user', 'content': [{'text': 'i want to go to a trip', 'type': 'text'}]}
4 {'role': 'assistant', 'content': [{'text': 'Sure, I can help with that. Do you have your destination in mind?', 'type': 'text'}]}
5 {'role': 'user', 'content': 'to london please'}
6 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_wk9bfqr4', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_pri

# Let's use lite database insted of hard coded data of ticket prices

In [13]:
import sqlite3

In [14]:
# this will create one file prices.db

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [12]:
# now lets update our get ticket price function 
# old
# def get_ticket_price(destination_city):
#     print(f"tool called for {destination_city} city")
#     price = ticket_prices.get(destination_city.lower(),'Unknown ticket price')
#     return f" the price of a ticket to {destination_city} is {price}"

# new
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else "No price data available for this city"


In [15]:
get_ticket_price('london')

DATABASE TOOL CALLED: Getting price for london


'Ticket price to london is $799.0'

In [16]:
# while getting above data is empty so lets set data first 

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [17]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [18]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [20]:
gr.ChatInterface(fn=chat_improved).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Model requested tool call
DATABASE TOOL CALLED: Getting price for london
DATABASE TOOL CALLED: Getting price for Berlin

Messages being sent back:

0 {'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
1 {'role': 'user', 'content': [{'text': 'hello', 'type': 'text'}]}
2 {'role': 'assistant', 'content': [{'text': "Hello! How can I help you today? Whether it's booking a ticket or finding out about pricing, just let me know.", 'type': 'text'}]}
3 {'role': 'user', 'content': 'what is the ticket price to london and Berlin'}
4 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_8xgbx0l5', function=Function(arguments='{"destination_city":"london"}', name='get_ticket_price'), type='function', index=0), ChatCompleti